# Data Preprocessing for MovieLens, Netflix, and Douban_Monti

This notebook preprocesses interaction-heavy datasets for the P5 framework:
- **ML-1M**: MovieLens 1M (1 million ratings)
- **ML-20M**: MovieLens 20M (20 million ratings)
- **Netflix**: Netflix Prize dataset
- **Douban_Monti**: Douban movie ratings (Monti et al.)

These datasets are text-light but interaction-heavy, making them ideal for:
- Rating prediction (Task Family 1)
- Sequential recommendation (Task Family 2)
- Direct recommendation (Task Family 5)

### Pretraining Strategy
1. **Pretrain** P5 on ML-20M + Netflix (large-scale semantic robustness)
2. **Add** Douban_Monti for cultural/domain shift
3. **Fine-tune** or prompt-adapt on ML-1M (faster convergence, cleaner evaluation)

In [ ]:
from collections import defaultdict
import os
import torch
import random
import numpy as np
import pandas as pd
import json
import pickle
from tqdm import tqdm

def load_pickle(filename):
    with open(filename, "rb") as f:
        return pickle.load(f)

def save_pickle(data, filename):
    with open(filename, "wb") as f:
        pickle.dump(data, f, protocol=pickle.HIGHEST_PROTOCOL)

def load_json(file_path):
    with open(file_path, "r") as f:
        return json.load(f)

def ReadLineFromFile(path):
    lines = []
    with open(path, 'r') as fd:
        for line in fd:
            lines.append(line.rstrip('\n'))
    return lines

'''
Set seeds
'''
seed = 2022
torch.manual_seed(seed)
random.seed(seed)
np.random.seed(seed)

## Configuration

Set which dataset to preprocess. Options:
- `ml-1m`
- `ml-20m`
- `netflix`
- `douban_monti`

In [ ]:
# Change this to process different datasets
dataset_name = 'ml-1m'  # Options: 'ml-1m', 'ml-20m', 'netflix', 'douban_monti'
os.makedirs(dataset_name, exist_ok=True)

## Dataset Loading Functions

Each dataset has its own format. We unify them into:
`(user_raw_id, item_raw_id, rating, timestamp)`

In [ ]:
def load_ml1m(rating_score=0.0):
    """
    Load MovieLens 1M dataset.
    Format: UserID::MovieID::Rating::Timestamp
    """
    datas = []
    data_file = './raw_data/ml-1m/ratings.dat'
    with open(data_file, 'r') as f:
        for line in f:
            user, item, rating, timestamp = line.strip().split('::')
            if float(rating) <= rating_score:
                continue
            datas.append((user, item, float(rating), int(timestamp)))
    return datas


def load_ml20m(rating_score=0.0):
    """
    Load MovieLens 20M dataset.
    CSV format: userId,movieId,rating,timestamp
    """
    datas = []
    data_file = './raw_data/ml-20m/ratings.csv'
    with open(data_file, 'r') as f:
        header = f.readline()  # skip header
        for line in tqdm(f, desc='Loading ML-20M'):
            user, item, rating, timestamp = line.strip().split(',')
            if float(rating) <= rating_score:
                continue
            datas.append((user, item, float(rating), int(timestamp)))
    return datas


def load_netflix(rating_score=0.0):
    """
    Load Netflix Prize dataset.
    Format: Each movie block starts with 'MovieID:'
    followed by lines of 'UserID,Rating,Date'
    
    We convert dates to unix timestamps for ordering.
    """
    from datetime import datetime
    datas = []
    data_dir = './raw_data/netflix/training_set/'
    
    # Netflix data may also come as combined files
    combined_file = './raw_data/netflix/combined_data.txt'
    
    if os.path.exists(combined_file):
        files = [combined_file]
    elif os.path.exists(data_dir):
        files = sorted([os.path.join(data_dir, f) for f in os.listdir(data_dir) if f.endswith('.txt')])
    else:
        # Try combined_data_1.txt through combined_data_4.txt
        files = [f'./raw_data/netflix/combined_data_{i}.txt' for i in range(1, 5)
                 if os.path.exists(f'./raw_data/netflix/combined_data_{i}.txt')]
    
    current_movie = None
    for filepath in tqdm(files, desc='Loading Netflix files'):
        with open(filepath, 'r') as f:
            for line in f:
                line = line.strip()
                if line.endswith(':'):
                    current_movie = line[:-1]  # Remove trailing ':'
                else:
                    parts = line.split(',')
                    if len(parts) >= 3:
                        user = parts[0]
                        rating = float(parts[1])
                        date_str = parts[2]
                        if rating <= rating_score:
                            continue
                        try:
                            timestamp = int(datetime.strptime(date_str, '%Y-%m-%d').timestamp())
                        except:
                            timestamp = 0
                        datas.append((user, current_movie, rating, timestamp))
    return datas


def load_douban_monti(rating_score=0.0):
    """
    Load Douban Monti dataset.
    Expected formats (tries multiple):
    1. CSV: user_id,item_id,rating (no timestamp - we assign sequential order)
    2. .mat file from Monti et al.
    3. TSV/space-separated: user_id item_id rating [timestamp]
    """
    datas = []
    
    # Try CSV format first
    csv_file = './raw_data/douban_monti/ratings.csv'
    tsv_file = './raw_data/douban_monti/ratings.tsv'
    dat_file = './raw_data/douban_monti/ratings.dat'
    mat_file = './raw_data/douban_monti/douban_monti.mat'
    
    if os.path.exists(csv_file):
        df = pd.read_csv(csv_file)
        # Normalize column names
        cols = df.columns.str.lower()
        df.columns = cols
        user_col = [c for c in cols if 'user' in c][0] if any('user' in c for c in cols) else cols[0]
        item_col = [c for c in cols if 'item' in c or 'movie' in c][0] if any('item' in c or 'movie' in c for c in cols) else cols[1]
        rating_col = [c for c in cols if 'rating' in c or 'score' in c][0] if any('rating' in c or 'score' in c for c in cols) else cols[2]
        timestamp_col = [c for c in cols if 'time' in c or 'stamp' in c]
        
        for idx, row in tqdm(df.iterrows(), total=len(df), desc='Loading Douban_Monti'):
            rating = float(row[rating_col])
            if rating <= rating_score:
                continue
            user = str(row[user_col])
            item = str(row[item_col])
            ts = int(row[timestamp_col[0]]) if timestamp_col else idx
            datas.append((user, item, rating, ts))
    
    elif os.path.exists(tsv_file):
        with open(tsv_file, 'r') as f:
            for idx, line in enumerate(tqdm(f, desc='Loading Douban_Monti')):
                parts = line.strip().split()
                if len(parts) >= 3:
                    user, item = parts[0], parts[1]
                    rating = float(parts[2])
                    if rating <= rating_score:
                        continue
                    ts = int(parts[3]) if len(parts) > 3 else idx
                    datas.append((user, item, rating, ts))
    
    elif os.path.exists(dat_file):
        with open(dat_file, 'r') as f:
            for idx, line in enumerate(tqdm(f, desc='Loading Douban_Monti')):
                parts = line.strip().split('::')
                if len(parts) < 3:
                    parts = line.strip().split(',')
                if len(parts) < 3:
                    parts = line.strip().split()
                user, item = parts[0], parts[1]
                rating = float(parts[2])
                if rating <= rating_score:
                    continue
                ts = int(parts[3]) if len(parts) > 3 else idx
                datas.append((user, item, rating, ts))
    
    elif os.path.exists(mat_file):
        import scipy.io as sio
        mat = sio.loadmat(mat_file)
        # Monti format typically has a sparse rating matrix
        if 'M' in mat:
            from scipy.sparse import issparse
            M = mat['M']
            if issparse(M):
                M = M.tocoo()
                for i in range(len(M.row)):
                    user = str(M.row[i])
                    item = str(M.col[i])
                    rating = float(M.data[i])
                    if rating <= rating_score:
                        continue
                    datas.append((user, item, rating, i))
            else:
                rows, cols = np.nonzero(M)
                for i in range(len(rows)):
                    user = str(rows[i])
                    item = str(cols[i])
                    rating = float(M[rows[i], cols[i]])
                    if rating <= rating_score:
                        continue
                    datas.append((user, item, rating, i))
    else:
        raise FileNotFoundError(
            f'No Douban_Monti data found. Please place data in one of: '
            f'{csv_file}, {tsv_file}, {dat_file}, or {mat_file}'
        )
    return datas

## Movie Metadata Loading

Load movie titles and genres where available.

In [ ]:
def load_ml1m_metadata():
    """
    Load ML-1M movie metadata.
    Format: MovieID::Title::Genres
    """
    meta = {}
    data_file = './raw_data/ml-1m/movies.dat'
    with open(data_file, 'r', encoding='latin-1') as f:
        for line in f:
            parts = line.strip().split('::')
            movie_id = parts[0]
            title = parts[1]
            genres = parts[2].split('|') if len(parts) > 2 else []
            meta[movie_id] = {'title': title, 'genres': genres}
    return meta


def load_ml20m_metadata():
    """
    Load ML-20M movie metadata.
    CSV format: movieId,title,genres
    """
    meta = {}
    data_file = './raw_data/ml-20m/movies.csv'
    df = pd.read_csv(data_file)
    for _, row in df.iterrows():
        movie_id = str(row['movieId'])
        title = str(row['title'])
        genres = str(row['genres']).split('|') if pd.notna(row['genres']) else []
        meta[movie_id] = {'title': title, 'genres': genres}
    return meta


def load_netflix_metadata():
    """
    Load Netflix movie metadata.
    Format: MovieID,YearOfRelease,Title
    """
    meta = {}
    data_file = './raw_data/netflix/movie_titles.csv'
    if not os.path.exists(data_file):
        data_file = './raw_data/netflix/movie_titles.txt'
    if os.path.exists(data_file):
        with open(data_file, 'r', encoding='latin-1') as f:
            for line in f:
                parts = line.strip().split(',', 2)
                if len(parts) >= 3:
                    movie_id = parts[0]
                    year = parts[1]
                    title = parts[2]
                    meta[movie_id] = {'title': title, 'year': year, 'genres': []}
    return meta


def load_douban_metadata():
    """
    Load Douban_Monti movie metadata if available.
    Returns empty dict if no metadata file found.
    """
    meta = {}
    for fname in ['movies.csv', 'items.csv', 'movie_info.csv']:
        data_file = f'./raw_data/douban_monti/{fname}'
        if os.path.exists(data_file):
            df = pd.read_csv(data_file)
            cols = df.columns.str.lower()
            df.columns = cols
            id_col = [c for c in cols if 'id' in c or 'item' in c or 'movie' in c][0] if any('id' in c or 'item' in c or 'movie' in c for c in cols) else cols[0]
            title_col = [c for c in cols if 'title' in c or 'name' in c]
            genre_col = [c for c in cols if 'genre' in c]
            for _, row in df.iterrows():
                item_id = str(row[id_col])
                title = str(row[title_col[0]]) if title_col else f'movie_{item_id}'
                genres = str(row[genre_col[0]]).split('|') if genre_col and pd.notna(row[genre_col[0]]) else []
                meta[item_id] = {'title': title, 'genres': genres}
            break
    return meta

## Core Processing Functions

Same K-core filtering and ID mapping as the Amazon/Yelp preprocessing.

In [ ]:
def get_interaction(datas):
    """Build user sequences sorted by timestamp."""
    user_seq = {}
    user_ratings = {}  # Also track ratings for each interaction
    for data in datas:
        user, item, rating, time = data
        if user in user_seq:
            user_seq[user].append((item, time, rating))
        else:
            user_seq[user] = [(item, time, rating)]

    for user, item_time_rating in user_seq.items():
        item_time_rating.sort(key=lambda x: x[1])  # Sort by timestamp
        items = [t[0] for t in item_time_rating]
        ratings = {t[0]: t[2] for t in item_time_rating}
        user_seq[user] = items
        user_ratings[user] = ratings
    return user_seq, user_ratings


def check_Kcore(user_items, user_core, item_core):
    user_count = defaultdict(int)
    item_count = defaultdict(int)
    for user, items in user_items.items():
        for item in items:
            user_count[user] += 1
            item_count[item] += 1
    for user, num in user_count.items():
        if num < user_core:
            return user_count, item_count, False
    for item, num in item_count.items():
        if num < item_core:
            return user_count, item_count, False
    return user_count, item_count, True


def filter_Kcore(user_items, user_core, item_core):
    user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    while not isKcore:
        for user, num in list(user_count.items()):
            if user_count[user] < user_core:
                if user in user_items:
                    user_items.pop(user)
            else:
                if user in user_items:
                    user_items[user] = [item for item in user_items[user] if item_count[item] >= item_core]
        # Remove empty users
        user_items = {u: items for u, items in user_items.items() if len(items) > 0}
        user_count, item_count, isKcore = check_Kcore(user_items, user_core, item_core)
    return user_items


def id_map(user_items):
    user2id = {}
    item2id = {}
    id2user = {}
    id2item = {}
    user_id = 1
    item_id = 1
    final_data = {}
    random_user_list = list(user_items.keys())
    random.shuffle(random_user_list)
    for user in random_user_list:
        items = user_items[user]
        if user not in user2id:
            user2id[user] = str(user_id)
            id2user[str(user_id)] = user
            user_id += 1
        iids = []
        for item in items:
            if item not in item2id:
                item2id[item] = str(item_id)
                id2item[str(item_id)] = item
                item_id += 1
            iids.append(item2id[item])
        uid = user2id[user]
        final_data[uid] = iids
    data_maps = {
        'user2id': user2id,
        'item2id': item2id,
        'id2user': id2user,
        'id2item': id2item
    }
    return final_data, user_id - 1, item_id - 1, data_maps


def add_comma(num):
    str_num = str(num)
    res_num = ''
    for i in range(len(str_num)):
        res_num += str_num[i]
        if (len(str_num) - i - 1) % 3 == 0:
            res_num += ','
    return res_num[:-1]

## Genre/Attribute Extraction

In [ ]:
def get_attributes_from_metadata(meta_infos, datamaps, attribute_core=0):
    """
    Extract genre/attribute information from movie metadata.
    Works with ML-1M, ML-20M, Netflix, and Douban_Monti.
    """
    attributes = defaultdict(int)
    for raw_item_id, info in meta_infos.items():
        if raw_item_id not in datamaps['item2id']:
            continue
        for genre in info.get('genres', []):
            if genre and genre != '(no genres listed)':
                attributes[genre] += 1

    print(f'Before filtering, attribute num: {len(attributes)}')
    
    attribute2id = {}
    id2attribute = {}
    attributeid2num = defaultdict(int)
    attribute_id = 1
    items2attributes = {}
    attribute_lens = []

    for raw_item_id, info in meta_infos.items():
        if raw_item_id not in datamaps['item2id']:
            continue
        item_id = datamaps['item2id'][raw_item_id]
        items2attributes[item_id] = []
        for genre in info.get('genres', []):
            if genre and genre != '(no genres listed)' and attributes[genre] >= attribute_core:
                if genre not in attribute2id:
                    attribute2id[genre] = attribute_id
                    id2attribute[attribute_id] = genre
                    attribute_id += 1
                attributeid2num[attribute2id[genre]] += 1
                items2attributes[item_id].append(attribute2id[genre])
        attribute_lens.append(len(items2attributes[item_id]))

    print(f'After filtering, attribute num: {len(attribute2id)}')
    if attribute_lens:
        print(f'Attributes len, Min:{np.min(attribute_lens)}, Max:{np.max(attribute_lens)}, Avg.:{np.mean(attribute_lens):.4f}')

    datamaps['attribute2id'] = attribute2id
    datamaps['id2attribute'] = id2attribute
    datamaps['attributeid2num'] = attributeid2num
    return len(attribute2id), np.mean(attribute_lens) if attribute_lens else 0, datamaps, items2attributes

## Main Processing Pipeline

In [ ]:
def main(dataset_name):
    rating_score = 0.0  # ratings <= this score are deleted
    user_core = 5
    item_core = 5
    attribute_core = 0

    # Load raw data
    print(f'Loading {dataset_name} raw data...')
    if dataset_name == 'ml-1m':
        datas = load_ml1m(rating_score)
        meta_infos = load_ml1m_metadata()
    elif dataset_name == 'ml-20m':
        datas = load_ml20m(rating_score)
        meta_infos = load_ml20m_metadata()
    elif dataset_name == 'netflix':
        datas = load_netflix(rating_score)
        meta_infos = load_netflix_metadata()
    elif dataset_name == 'douban_monti':
        datas = load_douban_monti(rating_score)
        meta_infos = load_douban_metadata()
    else:
        raise NotImplementedError(f'Unknown dataset: {dataset_name}')

    print(f'Loaded {len(datas)} raw interactions')

    # Build user sequences and apply K-core filtering
    user_items, user_ratings = get_interaction(datas)
    print(f'{dataset_name} Raw data has been processed! Lower than {rating_score} are deleted!')
    
    user_items = filter_Kcore(user_items, user_core=user_core, item_core=item_core)
    print(f'User {user_core}-core complete! Item {item_core}-core complete!')

    # ID mapping
    user_items, user_num, item_num, data_maps = id_map(user_items)
    user_count, item_count, _ = check_Kcore(user_items, user_core=user_core, item_core=item_core)
    user_count_list = list(user_count.values())
    user_avg = np.mean(user_count_list)
    user_min = np.min(user_count_list)
    user_max = np.max(user_count_list)
    item_count_list = list(item_count.values())
    item_avg = np.mean(item_count_list)
    item_min = np.min(item_count_list)
    item_max = np.max(item_count_list)
    interact_num = np.sum([x for x in user_count_list])
    sparsity = (1 - interact_num / (user_num * item_num)) * 100
    show_info = (
        f'Total User: {user_num}, Avg User: {user_avg:.4f}, Min Len: {user_min}, Max Len: {user_max}\n'
        f'Total Item: {item_num}, Avg Item: {item_avg:.4f}, Min Inter: {item_min}, Max Inter: {item_max}\n'
        f'Interaction Num: {interact_num}, Sparsity: {sparsity:.2f}%'
    )
    print(show_info)

    # Extract attributes from metadata
    print('\nExtracting metadata/genre attributes...')
    attribute_num, avg_attribute, data_maps, item2attributes = get_attributes_from_metadata(
        meta_infos, data_maps, attribute_core
    )

    print(f'\n{dataset_name} & {add_comma(user_num)}& {add_comma(item_num)} & {user_avg:.1f}'
          f'& {item_avg:.1f}& {add_comma(interact_num)}& {sparsity:.2f}\\%&{add_comma(attribute_num)}&'
          f'{avg_attribute:.1f} \\\\')

    # Save processed data
    data_file = f'./{dataset_name}/sequential_data.txt'
    item2attributes_file = f'./{dataset_name}/item2attributes.json'
    datamaps_file = f'./{dataset_name}/datamaps.json'

    with open(data_file, 'w') as out:
        for user, items in user_items.items():
            out.write(user + ' ' + ' '.join(items) + '\n')

    json_str = json.dumps(item2attributes)
    with open(item2attributes_file, 'w') as out:
        out.write(json_str)

    json_str = json.dumps(data_maps)
    with open(datamaps_file, 'w') as out:
        out.write(json_str)

    # Save movie title metadata for P5 templates
    item_id2title = {}
    for raw_id, info in meta_infos.items():
        if raw_id in data_maps['item2id']:
            mapped_id = data_maps['item2id'][raw_id]
            item_id2title[mapped_id] = info.get('title', f'item_{mapped_id}')
    save_pickle(item_id2title, f'./{dataset_name}/item_id2title.pkl')

    # Create user_id2name (use user_X format since MovieLens/Netflix have no user names)
    user_id2name = {}
    for uid in user_items.keys():
        user_id2name[uid] = f'user_{uid}'
    save_pickle(user_id2name, f'./{dataset_name}/user_id2name.pkl')

    return user_items, user_ratings, data_maps, meta_infos

In [ ]:
user_items, user_ratings, data_maps, meta_infos = main(dataset_name)

## Sample Negative Items for Evaluation

In [ ]:
def sample_test_data(data_name, test_num=99, sample_type='random'):
    """
    Sample `test_num` negative items per user for evaluation.
    """
    data_file = 'sequential_data.txt'
    test_file = 'negative_samples.txt'

    item_count = defaultdict(int)
    user_items = defaultdict()

    lines = open(f'./{data_name}/{data_file}').readlines()
    for line in lines:
        user, items = line.strip().split(' ', 1)
        items = items.split(' ')
        items = [int(item) for item in items]
        user_items[user] = items
        for item in items:
            item_count[item] += 1

    all_item = list(item_count.keys())
    count = list(item_count.values())
    sum_value = np.sum([x for x in count])
    probability = [value / sum_value for value in count]

    user_neg_items = defaultdict()

    for user, user_seq in tqdm(user_items.items(), desc='Sampling negatives'):
        test_samples = []
        while len(test_samples) < test_num:
            if sample_type == 'random':
                sample_ids = np.random.choice(all_item, test_num, replace=False)
            else:
                sample_ids = np.random.choice(all_item, test_num, replace=False, p=probability)
            sample_ids = [str(item) for item in sample_ids if item not in user_seq and item not in test_samples]
            test_samples.extend(sample_ids)
        test_samples = test_samples[:test_num]
        user_neg_items[user] = test_samples

    with open(f'./{data_name}/{test_file}', 'w') as out:
        for user, samples in user_neg_items.items():
            out.write(user + ' ' + ' '.join(samples) + '\n')

sample_test_data(dataset_name)

## Create Rating-Based Train/Val/Test Splits

Since these datasets are text-light (no reviews/explanations), we create splits
based on the rating interactions. The split follows:
- Train: 80%
- Val: 10%
- Test: 10%

We ensure every user and item appears at least once in the training set.

In [ ]:
# Build rating records from raw data
def build_rating_records(datas, data_maps):
    """
    Build rating records in P5-compatible format from raw interaction data.
    Each record contains: userID (mapped), itemID (mapped), overall (rating).
    """
    records = []
    for data in datas:
        user_raw, item_raw, rating, timestamp = data
        if user_raw in data_maps['user2id'] and item_raw in data_maps['item2id']:
            record = {
                'reviewerID': user_raw,
                'asin': item_raw,
                'overall': rating,
                'unixReviewTime': timestamp
            }
            records.append(record)
    return records

In [ ]:
# Reload raw data to build rating records
print(f'Reloading {dataset_name} for rating splits...')
if dataset_name == 'ml-1m':
    raw_datas = load_ml1m(0.0)
elif dataset_name == 'ml-20m':
    raw_datas = load_ml20m(0.0)
elif dataset_name == 'netflix':
    raw_datas = load_netflix(0.0)
elif dataset_name == 'douban_monti':
    raw_datas = load_douban_monti(0.0)

rating_records = build_rating_records(raw_datas, data_maps)
print(f'Total rating records after mapping: {len(rating_records)}')

In [ ]:
# Create train/val/test splits ensuring coverage
population = len(rating_records)
data = range(population)

user_mention_dict = {}
item_mention_dict = {}
for i in data:
    record = rating_records[i]
    user_ = record['reviewerID']
    item_ = record['asin']
    if user_ not in user_mention_dict:
        user_mention_dict[user_] = [i]
    else:
        user_mention_dict[user_].append(i)
    if item_ not in item_mention_dict:
        item_mention_dict[item_] = [i]
    else:
        item_mention_dict[item_].append(i)

# Ensure at least one record per user and item in train
train_indices = []
for u in tqdm(user_mention_dict.keys(), desc='Ensuring user coverage'):
    index_cand = user_mention_dict[u]
    random_choice = random.randint(0, len(index_cand) - 1)
    if index_cand[random_choice] not in train_indices:
        train_indices.append(index_cand[random_choice])
for it in tqdm(item_mention_dict.keys(), desc='Ensuring item coverage'):
    index_cand = item_mention_dict[it]
    random_choice = random.randint(0, len(index_cand) - 1)
    if index_cand[random_choice] not in train_indices:
        train_indices.append(index_cand[random_choice])
print(f'Minimum train indices (for coverage): {len(train_indices)}')

In [ ]:
remaining_indices = list(set(range(population)).difference(set(train_indices)))
print(f'Remaining indices: {len(remaining_indices)}')

# Fill up to 80% train
needed = round(population * 0.8) - len(train_indices)
if needed > 0:
    sub_indices = random.sample(range(len(remaining_indices)), min(needed, len(remaining_indices)))
    final_train_indices = list(set(np.array(remaining_indices)[sub_indices]).union(set(train_indices)))
else:
    final_train_indices = train_indices
print(f'Final train size: {len(final_train_indices)}')

# Val and test from remaining
val_test_indices = list(set(range(population)).difference(set(final_train_indices)))
print(f'Val+Test size: {len(val_test_indices)}')

val_size = round(population * 0.1)
sub_sub_indices = random.sample(range(len(val_test_indices)), min(val_size, len(val_test_indices)))
val_indices = list(np.array(val_test_indices)[sub_sub_indices])
test_indices = list(set(val_test_indices).difference(set(val_indices)))
print(f'Val size: {len(val_indices)}')
print(f'Test size: {len(test_indices)}')

all_indices = final_train_indices + val_indices + test_indices
print(f'Total (should match population {population}): {len(set(all_indices))}')

In [ ]:
# Build split data
train_review_data = [rating_records[i] for i in final_train_indices]
val_review_data = [rating_records[j] for j in val_indices]
test_review_data = [rating_records[k] for k in test_indices]

outputs = {
    'train': train_review_data,
    'val': val_review_data,
    'test': test_review_data,
    'train_indices': final_train_indices,
    'val_indices': val_indices,
    'test_indices': test_indices
}

save_pickle(outputs, f'./{dataset_name}/review_splits.pkl')
print(f'Saved review_splits.pkl (train: {len(train_review_data)}, val: {len(val_review_data)}, test: {len(test_review_data)})')

## Rating Re-balancing (Augmentation)

Same augmentation strategy as the Amazon preprocessing: balance minority rating classes
using Gaussian perturbation.

In [ ]:
# Check rating distribution
data_splits = load_pickle(f'./{dataset_name}/review_splits.pkl')
train_review_data = data_splits['train']

# Determine rating scale (Netflix/Douban may differ)
all_ratings_set = set()
for record in train_review_data:
    all_ratings_set.add(float(record['overall']))
all_ratings_sorted = sorted(all_ratings_set)
print(f'Rating values found: {all_ratings_sorted}')

# Use integer ratings 1-5 (standard for all four datasets)
all_ratings = [float(r) for r in range(1, 6)]

counts = {r: 0 for r in all_ratings}
for record in train_review_data:
    r = float(int(record['overall']))  # Round to integer
    if r in counts:
        counts[r] += 1
T = sum(counts.values())
print(f'Total training records: {T}')
for k, c in counts.items():
    counts[k] = float(c) / T if T > 0 else 0
print(f'Rating distribution: {counts}')

In [ ]:
import matplotlib.pyplot as plt
X = all_ratings
plt.bar(X, [counts[x] for x in X])
plt.xlabel('Rating')
plt.ylabel('Proportion')
plt.title(f'{dataset_name} - Training Rating Distribution (Before Augmentation)')
plt.show()

In [ ]:
# Augmentation
augment_prob = {r: max(1.0 / (len(counts) - 1) - c, 0.) for r, c in counts.items()}
sum_P = sum(augment_prob.values())
if sum_P > 0:
    augment_multiplier = {r: augment_prob[r] / (sum_P * counts[r]) if counts[r] > 0 else 0 for r in all_ratings}
else:
    augment_multiplier = {r: 0 for r in all_ratings}
print(f'Augment multiplier: {augment_multiplier}')

In [ ]:
# Augment with variation
dist_to_prob = [0.5, 0.1, 0.03, 0.01, 0.003]
rating_perturbation_prob = {}
for r in all_ratings:
    P = np.array([dist_to_prob[int(abs(r - r_2))] for r_2 in all_ratings])
    P /= np.sum(P)
    rating_perturbation_prob[r] = P

augmented_counts = {r: 0 for r in all_ratings}
augmented_records = []
for record in tqdm(train_review_data, desc='Augmenting ratings'):
    record_rating = float(int(record['overall']))
    if record_rating not in augment_multiplier:
        continue
    M = augment_multiplier[record_rating]
    remainder = M % 1
    M = int(M) + 1 if np.random.random() < remainder else int(M)
    if M == 0:
        continue
    sample_amount = np.random.multinomial(M, rating_perturbation_prob[record_rating])
    for i, n_sample in enumerate(sample_amount):
        for j in range(n_sample):
            new_record = {k: v for k, v in record.items()}
            new_record['overall'] = float(i + 1)
            augmented_records.append(new_record)
        augmented_counts[float(i + 1)] += n_sample

print(f'Augmented counts: {augmented_counts}')
print(f'Augmented records: {len(augmented_records)}')

In [ ]:
data_splits['train'] = data_splits['train'] + augmented_records
print(f'Total training records after augmentation: {len(data_splits["train"])}')

save_pickle(data_splits, f'./{dataset_name}/rating_splits_augmented.pkl')

In [ ]:
# Verify new distribution
train_review_data = data_splits['train']
counts_new = {r: 0 for r in all_ratings}
for record in train_review_data:
    r = float(int(record['overall']))
    if r in counts_new:
        counts_new[r] += 1
T_new = sum(counts_new.values())
for k, c in counts_new.items():
    counts_new[k] = float(c) / T_new
print(f'New distribution: {counts_new}')

plt.bar(X, [counts_new[x] for x in X])
plt.xlabel('Rating')
plt.ylabel('Proportion')
plt.title(f'{dataset_name} - Training Rating Distribution (After Augmentation)')
plt.show()

## Create Placeholder Explanation Splits

Since these datasets have no review text or explanations, we create empty explanation splits
to maintain compatibility with the P5 framework. The training will focus on:
- Rating prediction (Task Family 1)
- Sequential recommendation (Task Family 2)
- Direct recommendation (Task Family 5)

In [ ]:
# Create empty exp_splits for compatibility
exp_outputs = {
    'train': [],
    'val': [],
    'test': []
}
save_pickle(exp_outputs, f'./{dataset_name}/exp_splits.pkl')
print('Saved empty exp_splits.pkl (no review text available for these datasets)')
print(f'\nPreprocessing complete for {dataset_name}!')
print(f'Output files in ./{dataset_name}/:')
print('  - sequential_data.txt')
print('  - negative_samples.txt')
print('  - datamaps.json')
print('  - item2attributes.json')
print('  - item_id2title.pkl')
print('  - user_id2name.pkl')
print('  - review_splits.pkl')
print('  - rating_splits_augmented.pkl')
print('  - exp_splits.pkl')

## Batch Processing All Datasets

Uncomment and run the cell below to preprocess all four datasets sequentially.

In [ ]:
# # Batch process all datasets
# for ds in ['ml-1m', 'ml-20m', 'netflix', 'douban_monti']:
#     print(f'\n{"="*60}')
#     print(f'Processing {ds}...')
#     print(f'{"="*60}')
#     dataset_name = ds
#     os.makedirs(dataset_name, exist_ok=True)
#     user_items, user_ratings, data_maps, meta_infos = main(dataset_name)
#     sample_test_data(dataset_name)
#     # Then run the rating splits and augmentation cells above for each dataset